# Experiment Notebook: L2 Writing Error Detection and Error Type Classification

This notebook runs the Qwen3-8B experiments for the capstone project.

It includes:

1. **Task 1: Error Detection**  
   Predict whether the marked span is an actual error. Output: `0` or `1`.

2. **Task 2: Error Type Classification**  
   Predict three labels for the marked span. Output order: `expression,discourse,meaning`.

Task 2 includes three prompt conditions:

- `zero_shot_original`: original zero-shot prompt
- `constructed_few_shot`: few-shot examples dynamically sampled from the labeled dataset
- `human_prompt_zero_shot`: human-annotator-style prompt from the annotation guideline

Notes:

- The original dataset column name uses `sentence_with_boundry`. This notebook keeps that spelling for compatibility.
- If you use a different dataset path, update the paths in the configuration cell.

In [ ]:
# Install dependencies if needed.
# For a cluster/Jupyter environment, you may prefer to run this in terminal instead.

# !pip install -U "git+https://github.com/huggingface/transformers.git"
# !pip install -U accelerate sentencepiece pandas safetensors scikit-learn tqdm

In [ ]:
import os
import re
import json
import random
from pathlib import Path

import pandas as pd
import torch
import transformers
import accelerate
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)
from transformers import AutoTokenizer, AutoModelForCausalLM

print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("accelerate:", accelerate.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

## 1. Configuration

Update `BASE_DIR` if your files are in a different location.

Expected dataset columns:

Task 1:
- `sentence_with_boundry`
- `error_span`
- `gold_error`

Task 2:
- `sentence_with_boundry`
- `error_span`
- `gold_expression`
- `gold_discourse`
- `gold_meaning`

In [ ]:
# =========================
# Configuration
# =========================

BASE_DIR = Path("/home/jovyan/l2_error_expetiment/")

# If Task 1 and Task 2 are stored as separate files, set them here.
# If both tasks use the same full aggregated dataset, keep both paths the same.
TASK1_DATA_PATH = BASE_DIR / "task1_dataset.csv"
TASK2_DATA_PATH = BASE_DIR / "task2_dataset.csv"

# Fallback to the full aggregated dataset used in the original notebooks.
FULL_DATA_PATH = BASE_DIR / "full_aggregated_dataset.csv"
if not TASK1_DATA_PATH.exists() and FULL_DATA_PATH.exists():
    TASK1_DATA_PATH = FULL_DATA_PATH
if not TASK2_DATA_PATH.exists() and FULL_DATA_PATH.exists():
    TASK2_DATA_PATH = FULL_DATA_PATH

RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "Qwen/Qwen3-8B"
RANDOM_STATE = 42

# Run switches. Set False for conditions you do not want to run.
RUN_TASK1_ZERO_SHOT = True
RUN_TASK2_ZERO_SHOT_ORIGINAL = True
RUN_TASK2_CONSTRUCTED_FEW_SHOT = True
RUN_TASK2_HUMAN_PROMPT_ZERO_SHOT = True

# Optional: use a small number for testing, or None for the full dataset.
MAX_ROWS_TASK1 = None
MAX_ROWS_TASK2 = None

print("Task 1 data:", TASK1_DATA_PATH)
print("Task 2 data:", TASK2_DATA_PATH)
print("Results dir:", RESULTS_DIR)

In [ ]:
# =========================
# Load data
# =========================

task1_df = pd.read_csv(TASK1_DATA_PATH)
task2_df = pd.read_csv(TASK2_DATA_PATH)

if MAX_ROWS_TASK1 is not None:
    task1_df = task1_df.head(MAX_ROWS_TASK1).copy()
if MAX_ROWS_TASK2 is not None:
    task2_df = task2_df.head(MAX_ROWS_TASK2).copy()

print("Task 1 shape:", task1_df.shape)
print("Task 2 shape:", task2_df.shape)
print("Task 1 columns:", task1_df.columns.tolist())
print("Task 2 columns:", task2_df.columns.tolist())

TASK1_REQUIRED = ["sentence_with_boundry", "error_span", "gold_error"]
TASK2_REQUIRED = [
    "sentence_with_boundry",
    "error_span",
    "gold_expression",
    "gold_discourse",
    "gold_meaning",
]

def check_required_columns(df, required, task_name):
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{task_name} missing required columns: {missing}")

check_required_columns(task1_df, TASK1_REQUIRED, "Task 1")
check_required_columns(task2_df, TASK2_REQUIRED, "Task 2")

In [ ]:
# =========================
# Load model
# =========================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

print("Loaded model:", MODEL_NAME)

## 2. Output cleaning and generation helpers

In [ ]:
def clean_output_binary(text):
    """Clean model output for binary labels: 0 or 1."""
    text = str(text).strip()
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    text = re.sub(r"<think>.*", "", text, flags=re.DOTALL)
    text = re.sub(r"^(Answer:|Output:|Prediction:)\s*", "", text, flags=re.IGNORECASE)

    lines = [line.strip() for line in text.split("\n") if line.strip()]
    text = lines[0] if lines else ""
    text = text.strip().strip('"').strip("'")

    match = re.search(r"\b([01])\b", text)
    return match.group(1) if match else text


def clean_output_task2(text):
    """Clean model output for Task 2 multilabel output: expression,discourse,meaning."""
    text = str(text).strip()
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    text = re.sub(r"<think>.*", "", text, flags=re.DOTALL)
    text = re.sub(r"^(Answer:|Output:|Prediction:)\s*", "", text, flags=re.IGNORECASE)

    lines = [line.strip() for line in text.split("\n") if line.strip()]
    text = lines[0] if lines else ""
    text = text.strip().strip('"').strip("'")

    m = re.search(r"([01])\s*,\s*([01])\s*,\s*([01])", text)
    if not m:
        return None

    return {
        "expression": int(m.group(1)),
        "discourse": int(m.group(2)),
        "meaning": int(m.group(3)),
    }


def generate_from_prompt(prompt, max_new_tokens=8):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    return response

## 3. Prompt definitions

In [ ]:
def build_task1_zero_shot_prompt(sentence_with_boundry, error_span):
    return f"""/no_think

You are annotating pragmatic errors in L2 English writing.

A sentence and a marked error span are given.

Task:
Decide whether the marked span is actually an error in the sentence.

Definitions:
- Expression: awkward or inappropriate wording or collocation
- Discourse: issue in structure, flow, or discourse relation
- Meaning: unclear, misleading, or pragmatically inappropriate meaning

Sentence with marked span:
{sentence_with_boundry}

Error span:
{error_span}

Rules:
- Judge only whether the marked span is an actual error in this sentence.
- Output only one number.
- 1 = yes, the marked span is an error
- 0 = no, the marked span is not an error
- Do NOT explain anything.
- Output ONLY 0 or 1.
"""


def build_task2_zero_shot_original_prompt(sentence_with_boundry, error_span):
    return f"""/no_think

You are annotating pragmatic error types in L2 English writing.

A sentence and a marked error span are given.

Task:
Classify which pragmatic error types apply to the marked span.

Definitions:
- Expression: awkward or inappropriate wording or collocation
- Discourse: issue in structure, flow, or discourse relation
- Meaning: unclear, misleading, or pragmatically inappropriate meaning

Sentence with marked span:
{sentence_with_boundry}

Error span:
{error_span}

Rules:
- Judge the marked span only.
- This is a multi-label task.
- Output three binary numbers in this exact order:
expression,discourse,meaning
- Example: 1,0,1
- Do NOT explain anything.
- Output ONLY the three numbers.
"""

In [ ]:
TASK2_LABEL_DEFS = """
You are annotating pragmatic error types in L2 English writing.

A sentence and a marked error span are given.

Classify whether the marked span has each of the following labels.

Label definitions:

Expression
- The main problem is local wording.
- This includes awkward phrase choice, unnatural collocation, wrong lexical choice, or non-native-like wording.
- The sentence is still mostly understandable.
- A small local revision would usually fix it.

Discourse
- The main problem is how ideas are connected or organized.
- This includes incorrect logical connection, broken clause relation, confusing sentence structure across clauses, or redundancy of information.
- Do NOT use Discourse for simple grammar, spelling, punctuation, or local word choice problems.

Meaning
- The main problem is that the intended meaning is unclear, misleading, or hard to recover.
- Even if local words are corrected, the message is still difficult to understand.
- Use Meaning when the reader cannot confidently infer what the writer is trying to say.

Output format:
Return exactly three binary numbers in this order:
expression,discourse,meaning

Example:
1,0,1
"""


def sample_fewshot_excluding_current(df, current_row, target_label="expression", n_pos=2, n_neg=2, random_state=42):
    """Construct few-shot examples from gold-labeled data while excluding the current row."""
    pos_col = f"gold_{target_label}"

    pool_df = df.copy()
    pool_df = pool_df[pool_df["sentence_with_boundry"] != current_row["sentence_with_boundry"]]

    if "error_span" in pool_df.columns:
        pool_df = pool_df[pool_df["error_span"] != current_row["error_span"]]

    pos_candidates = pool_df[pool_df[pos_col] == 1]
    neg_candidates = pool_df[pool_df[pos_col] == 0]

    pos_df = pos_candidates.sample(n=min(n_pos, len(pos_candidates)), random_state=random_state)
    neg_df = neg_candidates.sample(n=min(n_neg, len(neg_candidates)), random_state=random_state)

    fewshot_df = pd.concat([pos_df, neg_df])
    if len(fewshot_df) == 0:
        return fewshot_df

    return fewshot_df.sample(frac=1, random_state=random_state).reset_index(drop=True)


def format_fewshot_block(example_row):
    return f"""Sentence with marked span:
{example_row['sentence_with_boundry']}

Error span:
{example_row['error_span']}

Output:
{int(example_row['gold_expression'])},{int(example_row['gold_discourse'])},{int(example_row['gold_meaning'])}
"""


def build_task2_constructed_fewshot_prompt(sentence_with_boundry, error_span, fewshot_rows):
    fewshot_text = "

".join([format_fewshot_block(row) for _, row in fewshot_rows.iterrows()])

    return f"""/no_think

{TASK2_LABEL_DEFS}

Here are some labeled examples:

{fewshot_text}

Now classify the next example.

Sentence with marked span:
{sentence_with_boundry}

Error span:
{error_span}

Output only the three binary numbers:
expression,discourse,meaning
"""

In [ ]:
HUMAN_ANNOTATOR_TASK2_PROMPT = """
You are annotating error labels in L2 English writing.

4. Error Labels
You will use the following three error labels.
A span can have multiple labels if needed.

Expression (Word Choice / Phrase)
Question:
Do the words sound unnatural or non-native?
Use when a phrase is understandable but sounds awkward or unnatural.
Examples:
“give bad influence” → have a negative influence
“make an experience” → gain experience
Reason: unnatural word or phrase choice

Discourse (Structure / Flow / Style)
Question:
Is the sentence structure, flow, or logic awkward?
Use when the sentence is structurally unnatural or poorly connected.
Examples:
“Because smoking is bad.” → incomplete clause
“But not graduate.” → sentence fragment
“you who do many jobs” → awkward structure
Reason: structure, logic, or style problem

Meaning (Unclear / Hard to Understand)
Question:
Is the meaning unclear or difficult to understand?
Use when the intended meaning is confusing or unclear.
Examples:
“She made herself own decision by himself” → pronoun confusion / unclear meaning

Task:
Given a sentence and a marked error span, decide which labels apply.

Output format:
Return exactly three binary numbers in this order:
expression,discourse,meaning

Example:
1,0,1
"""


def build_task2_human_prompt(sentence_with_boundry, error_span):
    return f"""/no_think

{HUMAN_ANNOTATOR_TASK2_PROMPT}

Sentence with marked span:
{sentence_with_boundry}

Error span:
{error_span}

Output only the three binary numbers:
expression,discourse,meaning
"""

## 4. Run experiments

In [ ]:
def run_task1_zero_shot(df):
    preds = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Task 1 zero-shot"):
        prompt = build_task1_zero_shot_prompt(row["sentence_with_boundry"], row["error_span"])
        raw_response = generate_from_prompt(prompt, max_new_tokens=6)
        cleaned = clean_output_binary(raw_response)
        pred = int(cleaned) if cleaned in ["0", "1"] else None

        preds.append({
            "prompt_condition": "zero_shot_original",
            "raw_response": raw_response,
            "cleaned_response": cleaned,
            "pred_error": pred,
        })

    out = df.copy()
    pred_df = pd.DataFrame(preds)
    return pd.concat([out.reset_index(drop=True), pred_df.reset_index(drop=True)], axis=1)


def run_task2_zero_shot_original(df):
    preds = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Task 2 zero-shot original"):
        prompt = build_task2_zero_shot_original_prompt(row["sentence_with_boundry"], row["error_span"])
        raw_response = generate_from_prompt(prompt, max_new_tokens=8)
        pred = clean_output_task2(raw_response)

        preds.append({
            "prompt_condition": "zero_shot_original",
            "raw_response": raw_response,
            "pred_expression": None if pred is None else pred["expression"],
            "pred_discourse": None if pred is None else pred["discourse"],
            "pred_meaning": None if pred is None else pred["meaning"],
        })

    out = df.copy()
    pred_df = pd.DataFrame(preds)
    return pd.concat([out.reset_index(drop=True), pred_df.reset_index(drop=True)], axis=1)


def run_task2_constructed_fewshot(df, target_label="expression", n_pos=2, n_neg=2):
    """
    Constructed few-shot condition.

    The few-shot examples are sampled from the same labeled dataset, excluding the current item.
    target_label controls the positive/negative balance used for example selection.
    """
    preds = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Task 2 constructed few-shot: {target_label}"):
        fewshot_rows = sample_fewshot_excluding_current(
            df,
            current_row=row,
            target_label=target_label,
            n_pos=n_pos,
            n_neg=n_neg,
            random_state=RANDOM_STATE,
        )
        prompt = build_task2_constructed_fewshot_prompt(row["sentence_with_boundry"], row["error_span"], fewshot_rows)
        raw_response = generate_from_prompt(prompt, max_new_tokens=8)
        pred = clean_output_task2(raw_response)

        preds.append({
            "prompt_condition": f"constructed_few_shot_{target_label}",
            "fewshot_target": target_label,
            "raw_response": raw_response,
            "pred_expression": None if pred is None else pred["expression"],
            "pred_discourse": None if pred is None else pred["discourse"],
            "pred_meaning": None if pred is None else pred["meaning"],
        })

    out = df.copy()
    pred_df = pd.DataFrame(preds)
    return pd.concat([out.reset_index(drop=True), pred_df.reset_index(drop=True)], axis=1)


def run_task2_human_prompt_zero_shot(df):
    preds = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Task 2 human prompt zero-shot"):
        prompt = build_task2_human_prompt(row["sentence_with_boundry"], row["error_span"])
        raw_response = generate_from_prompt(prompt, max_new_tokens=8)
        pred = clean_output_task2(raw_response)

        preds.append({
            "prompt_condition": "human_prompt_zero_shot",
            "raw_response": raw_response,
            "pred_expression": None if pred is None else pred["expression"],
            "pred_discourse": None if pred is None else pred["discourse"],
            "pred_meaning": None if pred is None else pred["meaning"],
        })

    out = df.copy()
    pred_df = pd.DataFrame(preds)
    return pd.concat([out.reset_index(drop=True), pred_df.reset_index(drop=True)], axis=1)

In [ ]:
# Run Task 1

if RUN_TASK1_ZERO_SHOT:
    task1_zero_df = run_task1_zero_shot(task1_df)
    task1_zero_path = RESULTS_DIR / "task1_qwen_zero_shot_original.csv"
    task1_zero_df.to_csv(task1_zero_path, index=False)
    print("Saved:", task1_zero_path)
else:
    task1_zero_df = None

In [ ]:
# Run Task 2 conditions

task2_outputs = {}

if RUN_TASK2_ZERO_SHOT_ORIGINAL:
    task2_outputs["zero_shot_original"] = run_task2_zero_shot_original(task2_df)
    path = RESULTS_DIR / "task2_qwen_zero_shot_original.csv"
    task2_outputs["zero_shot_original"].to_csv(path, index=False)
    print("Saved:", path)

if RUN_TASK2_CONSTRUCTED_FEW_SHOT:
    # This runs three constructed few-shot variants, one balanced around each target label.
    for target_label in ["expression", "discourse", "meaning"]:
        key = f"constructed_few_shot_{target_label}"
        task2_outputs[key] = run_task2_constructed_fewshot(
            task2_df,
            target_label=target_label,
            n_pos=2,
            n_neg=2,
        )
        path = RESULTS_DIR / f"task2_qwen_{key}.csv"
        task2_outputs[key].to_csv(path, index=False)
        print("Saved:", path)

if RUN_TASK2_HUMAN_PROMPT_ZERO_SHOT:
    task2_outputs["human_prompt_zero_shot"] = run_task2_human_prompt_zero_shot(task2_df)
    path = RESULTS_DIR / "task2_qwen_human_prompt_zero_shot.csv"
    task2_outputs["human_prompt_zero_shot"].to_csv(path, index=False)
    print("Saved:", path)

## 5. Evaluation

In [ ]:
def evaluate_task1(df):
    valid = df.dropna(subset=["pred_error"]).copy()
    valid["pred_error"] = valid["pred_error"].astype(int)
    valid["gold_error"] = valid["gold_error"].astype(int)

    y_true = valid["gold_error"]
    y_pred = valid["pred_error"]

    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)

    summary = pd.DataFrame([{
        "task": "task1_error_detection",
        "condition": df["prompt_condition"].iloc[0] if "prompt_condition" in df.columns else "unknown",
        "n_total": len(df),
        "n_valid": len(valid),
        "accuracy": acc,
        "precision": p,
        "recall": r,
        "f1": f1,
    }])

    print(summary)
    print("
Classification report:")
    print(classification_report(y_true, y_pred, zero_division=0))
    print("
Confusion matrix:")
    print(confusion_matrix(y_true, y_pred))

    return summary


def evaluate_task2_labels(df):
    rows = []
    condition = df["prompt_condition"].iloc[0] if "prompt_condition" in df.columns else "unknown"

    for label in ["expression", "discourse", "meaning"]:
        gold_col = f"gold_{label}"
        pred_col = f"pred_{label}"

        valid = df.dropna(subset=[pred_col]).copy()
        y_true = valid[gold_col].astype(int)
        y_pred = valid[pred_col].astype(int)

        p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
        acc = accuracy_score(y_true, y_pred)

        rows.append({
            "task": "task2_error_type_classification",
            "condition": condition,
            "label": label,
            "n_total": len(df),
            "n_valid": len(valid),
            "accuracy": acc,
            "precision": p,
            "recall": r,
            "f1": f1,
        })

    result_df = pd.DataFrame(rows)
    macro_row = {
        "task": "task2_error_type_classification",
        "condition": condition,
        "label": "macro_average",
        "n_total": len(df),
        "n_valid": result_df["n_valid"].min(),
        "accuracy": result_df["accuracy"].mean(),
        "precision": result_df["precision"].mean(),
        "recall": result_df["recall"].mean(),
        "f1": result_df["f1"].mean(),
    }
    result_df = pd.concat([result_df, pd.DataFrame([macro_row])], ignore_index=True)
    return result_df

In [ ]:
summary_tables = []

if task1_zero_df is not None:
    task1_summary = evaluate_task1(task1_zero_df)
    summary_tables.append(task1_summary)

for name, df in task2_outputs.items():
    eval_df = evaluate_task2_labels(df)
    print("
===", name, "===")
    print(eval_df)
    summary_tables.append(eval_df)

all_results_summary = pd.concat(summary_tables, ignore_index=True) if summary_tables else pd.DataFrame()
summary_path = RESULTS_DIR / "experiment_summary.csv"
all_results_summary.to_csv(summary_path, index=False)

print("
Saved summary:", summary_path)
all_results_summary

## 6. Save prompts for reproducibility

This creates a small prompt documentation file that can be included in the GitHub repository.

In [ ]:
prompts_doc = {
    "task1_zero_shot_original": build_task1_zero_shot_prompt("<sentence_with_boundry>", "<error_span>"),
    "task2_zero_shot_original": build_task2_zero_shot_original_prompt("<sentence_with_boundry>", "<error_span>"),
    "task2_constructed_few_shot_template": build_task2_constructed_fewshot_prompt(
        "<sentence_with_boundry>",
        "<error_span>",
        pd.DataFrame([
            {
                "sentence_with_boundry": "<few-shot sentence>",
                "error_span": "<few-shot error span>",
                "gold_expression": 1,
                "gold_discourse": 0,
                "gold_meaning": 1,
            }
        ]),
    ),
    "task2_human_prompt_zero_shot": build_task2_human_prompt("<sentence_with_boundry>", "<error_span>"),
}

prompts_path = RESULTS_DIR / "prompts_used.json"
with open(prompts_path, "w", encoding="utf-8") as f:
    json.dump(prompts_doc, f, ensure_ascii=False, indent=2)

print("Saved prompts:", prompts_path)